In [3]:
!pip install anthropic

   ---------------------------------------- 0.0/662.1 kB ? eta -:--:--
   ------------------------------- -------- 524.3/662.1 kB 3.4 MB/s eta 0:00:01
   ---------------------------------------- 662.1/662.1 kB 3.2 MB/s eta 0:00:00


In [1]:
import pandas as pd
import numpy as np
import os
import shutil
import time
import importlib.util
from datetime import datetime
import anthropic
from evaluate import evaluate  # immutable — agent never sees or touches this

# ── CONFIG ──────────────────────────────────────────
API_KEY        = os.environ.get("ANTHROPIC_API_KEY")
MODEL_NAME     = "claude-sonnet-4-6"
MAX_ITERATIONS = 300                            # set to 300 after test run
LOG_FILE       = "results_log.csv"
STRATEGY_FILE  = "strategy.py"
PROGRAM_FILE   = "program.md"
HISTORY_WINDOW = 10
FEE_RATE       = 0.001                        # 0.1% per trade
MIN_TRADES     = 30                         # reject strategies below this
# ────────────────────────────────────────────────────

client = anthropic.Anthropic(api_key=API_KEY)

# Test connection
test = client.messages.create(
    model=MODEL_NAME,
    max_tokens=10,
    messages=[{"role": "user", "content": "say only the word: connected"}]
)
print("Claude status:", test.content[0].text.strip())

Claude status: connected


In [3]:
def run_backtest():
    """
    Load strategy.py, run it on 2023 validation data,
    apply transaction costs, and return metrics.
    """
    df = pd.read_csv('data/btc_hourly.csv', parse_dates=['date'])
    val = df[(df['date'] >= '2023-01-01') & (df['date'] < '2024-01-01')].copy()

    spec = importlib.util.spec_from_file_location("strategy", STRATEGY_FILE)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)

    result = mod.run_strategy(val)

    # Verify strategy_returns column exists
    if "strategy_returns" not in result.columns:
        raise ValueError("strategy_returns column missing from run_strategy() output")

    # Count trades
    trades = result["signal"].diff().abs() > 0
    n_trades = int(trades.sum())
    turnover_per_day = round(n_trades / (len(result) / 24), 2)

    # Evaluate using immutable evaluate.py
    metrics = evaluate(result)
    metrics["n_trades"] = n_trades
    metrics["turnover_per_day"] = turnover_per_day

    return metrics

print("run_backtest() ready")

run_backtest() ready


In [5]:
def validate_code(code):
    """
    Returns (True, None) if code passes all guards.
    Returns (False, reason) if any guard is violated.
    """
    # Guard 1: forbidden imports
    forbidden = [
        "import scipy", "import sklearn", "import torch",
        "import tensorflow", "import statsmodels", "import talib",
        "import requests", "import urllib"
    ]
    for lib in forbidden:
        if lib in code:
            return False, f"forbidden import: {lib}"

    # Guard 2: agent must not redefine evaluate()
    if "def evaluate(" in code:
        return False, "agent redefined evaluate()"

    # Guard 3: strategy_returns must be present
    if "strategy_returns" not in code:
        return False, "strategy_returns column missing"

    # Guard 4: must start with pandas import
    if not code.strip().startswith("import pandas"):
        return False, "code does not start with import pandas"

    # Guard 5: syntax check
    try:
        compile(code, "<string>", "exec")
    except SyntaxError as e:
        return False, f"syntax error: {e}"

    # Guard 6: warnings import causes crashes inside run_strategy
    if "import warnings" in code:
        return False, "forbidden import: warnings"

    return True, None

print("validate_code() ready")

validate_code() ready


In [7]:
def build_history(n=HISTORY_WINDOW):
    """
    Build a short experiment history string to inject into the prompt.
    Gives the agent memory of what was already tried.
    """
    if not os.path.exists(LOG_FILE):
        return "No experiments yet."

    log = pd.read_csv(LOG_FILE)
    if len(log) == 0:
        return "No experiments yet."

    recent = log.tail(n)
    lines = []
    for _, row in recent.iterrows():
        status = "KEPT" if row['kept'] else "REVERTED"
        lines.append(
            f"  Iter {int(row['iteration'])}: "
            f"Sharpe={row['sharpe']} | "
            f"Return={row['total_return']}% | "
            f"Trades={row['n_trades']} | "
            f"{status} | {row['note']}"
        )
    return "\n".join(lines)

print("build_history() ready")

build_history() ready


In [9]:
def ask_agent(current_code, current_sharpe):
    """
    Send current strategy + history to Claude.
    Return the proposed new strategy code.
    """
    with open(PROGRAM_FILE, 'r') as f:
        prompt = f.read()

    prompt = prompt.replace('{sharpe}', str(current_sharpe))
    prompt = prompt.replace('{code}', current_code)
    prompt = prompt.replace('{history}', build_history())

    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=4096,
        messages=[{"role": "user", "content": prompt}]
    )
    new_code = response.content[0].text.strip()

    # Strip markdown fences if present
    if new_code.startswith("```"):
        lines = new_code.split('\n')
        lines = [l for l in lines if not l.startswith("```")]
        new_code = '\n'.join(lines).strip()

    time.sleep(3)  # Claude is faster than Gemini, less sleep needed
    return new_code

print("ask_agent() ready")

ask_agent() ready


In [11]:
def log_result(iteration, metrics, kept, note, code=None):
    """
    Append one row to results_log.csv.
    Save full strategy code to kept_strategies/ if kept.
    """
    row = {
        'iteration':         iteration,
        'timestamp':         datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'sharpe':            metrics.get('sharpe', None),
        'total_return':      metrics.get('total_return', None),
        'max_drawdown':      metrics.get('max_drawdown', None),
        'n_trades':          metrics.get('n_trades', None),
        'turnover_per_day':  metrics.get('turnover_per_day', None),
        'kept':              kept,
        'note':              note
    }

    df_log = pd.DataFrame([row])
    write_header = not os.path.exists(LOG_FILE)
    df_log.to_csv(LOG_FILE, mode='a', header=write_header, index=False)

    # Save full code for every accepted strategy
    if kept and code:
        os.makedirs("kept_strategies", exist_ok=True)
        fname = (
            f"kept_strategies/"
            f"strategy_iter{iteration}_"
            f"sharpe{metrics['sharpe']}.py"
        )
        with open(fname, 'w') as f:
            f.write(code)

print("log_result() ready")

log_result() ready


In [ ]:
# ── BASELINE ──────────────────────────────────────────
print("Running baseline backtest...")
try:
    baseline = run_backtest()
except Exception as e:
    print(f"Baseline failed: {e}")
    raise

best_sharpe = baseline['sharpe']
print(f"\n=== Baseline Results ===")
print(f"  Sharpe:          {best_sharpe}")
print(f"  Total Return:    {baseline['total_return']}%")
print(f"  Max Drawdown:    {baseline['max_drawdown']}%")
print(f"  Trades:          {baseline['n_trades']}")
print(f"  Turnover/day:    {baseline['turnover_per_day']}")

with open(STRATEGY_FILE, 'r') as f:
    best_code = f.read()

log_result(0, baseline, True, "baseline", best_code)

# ── MAIN LOOP ──────────────────────────────────────────
print(f"\nStarting loop — {MAX_ITERATIONS} iterations")
print("=" * 55)

for i in range(1, MAX_ITERATIONS + 1):
    print(f"\n[Iteration {i}/{MAX_ITERATIONS}]")

    # Step 1: ask agent
    print("  Asking agent for improvement...")
    try:
        new_code = ask_agent(best_code, best_sharpe)
    except Exception as e:
        print(f"  Agent error: {e} — skipping")
        log_result(i, {
            'sharpe': best_sharpe, 'total_return': None,
            'max_drawdown': None, 'n_trades': None,
            'turnover_per_day': None
        }, False, f"agent error: {e}")
        time.sleep(15)
        continue

    # Step 2: validate code guards
    valid, reason = validate_code(new_code)
    if not valid:
        print(f"  ❌ REJECTED by guard: {reason}")
        log_result(i, {
            'sharpe': best_sharpe, 'total_return': None,
            'max_drawdown': None, 'n_trades': None,
            'turnover_per_day': None
        }, False, f"guard rejection: {reason}")
        continue

    # Step 3: write new strategy and run backtest
    shutil.copy(STRATEGY_FILE, STRATEGY_FILE + '.backup')
    with open(STRATEGY_FILE, 'w') as f:
        f.write(new_code)

    print("  Running backtest...")
    try:
        metrics = run_backtest()
        new_sharpe = metrics['sharpe']
        print(f"  New Sharpe:      {new_sharpe}")
        print(f"  Best so far:     {best_sharpe}")
        print(f"  Trades:          {metrics['n_trades']} | "
              f"Turnover/day: {metrics['turnover_per_day']}")
    except Exception as e:
        print(f"  Backtest error: {e} — reverting")
        shutil.copy(STRATEGY_FILE + '.backup', STRATEGY_FILE)
        log_result(i, {
            'sharpe': best_sharpe, 'total_return': None,
            'max_drawdown': None, 'n_trades': None,
            'turnover_per_day': None
        }, False, f"backtest error: {e}")
        continue

    # Step 4: minimum trades guard
    if metrics['n_trades'] < MIN_TRADES:
        print(f"  ❌ REJECTED — too few trades "
              f"({metrics['n_trades']} < {MIN_TRADES})")
        shutil.copy(STRATEGY_FILE + '.backup', STRATEGY_FILE)
        log_result(i, metrics, False,
                   f"too few trades: {metrics['n_trades']}")
        continue

    # Step 5: keep or revert
    if new_sharpe > best_sharpe:
        best_sharpe = new_sharpe
        best_code = new_code
        print(f"  ✅ KEPT — Sharpe improved to {best_sharpe}")
        log_result(i, metrics, True, "improved", new_code)
    else:
        shutil.copy(STRATEGY_FILE + '.backup', STRATEGY_FILE)
        print(f"  ❌ REVERTED — no improvement")
        log_result(i, metrics, False, "no improvement")

# ── SUMMARY ───────────────────────────────────────────
print("\n" + "=" * 55)
print(f"Loop finished!")
print(f"Best Sharpe achieved: {best_sharpe}")
print(f"Results saved to:     {LOG_FILE}")
print(f"Kept strategies in:   kept_strategies/")

Running baseline backtest...


C:\Users\diell\OneDrive\Desktop\crypto-autoresearch\strategy.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  - trades.shift(1).fillna(False).astype(float) * FEE_RATE



=== Baseline Results ===
  Sharpe:          0.5646
  Total Return:    16.09%
  Max Drawdown:    -45.85%
  Trades:          228
  Turnover/day:    0.63

Starting loop — 300 iterations

[Iteration 1/300]
  Asking agent for improvement...
  Running backtest...


C:\Users\diell\OneDrive\Desktop\crypto-autoresearch\strategy.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


  New Sharpe:      -1.1552
  Best so far:     0.5646
  Trades:          672 | Turnover/day: 1.85
  ❌ REVERTED — no improvement

[Iteration 2/300]
  Asking agent for improvement...
  Running backtest...


...

C:\Users\diell\OneDrive\Desktop\crypto-autoresearch\strategy.py:100: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


  New Sharpe:      1.9977
  Best so far:     2.0046
  Trades:          44 | Turnover/day: 0.12
  ❌ REVERTED — no improvement

[Iteration 163/300]
  Asking agent for improvement...
  Running backtest...


C:\Users\diell\OneDrive\Desktop\crypto-autoresearch\strategy.py:100: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


  New Sharpe:      1.9977
  Best so far:     2.0046
  Trades:          44 | Turnover/day: 0.12
  ❌ REVERTED — no improvement

[Iteration 164/300]
  Asking agent for improvement...
  Agent error: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CaUys687HUrfYEcNX4khk'} — skipping

[Iteration 165/300]
  Asking agent for improvement...
  Agent error: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CaUytDvwYZKGykd7eNv4v'} — skipping

[Iteration 166/300]
  Asking agent for improvement...
  Agent error: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low

KeyboardInterrupt: 

In [15]:
import os

files = os.listdir(r"C:\Users\diell\OneDrive\Desktop\crypto-autoresearch\kept_strategies")
for f in files:
    print(f)

strategy_iter0_sharpe0.5646.py
strategy_iter11_sharpe1.732.py
strategy_iter124_sharpe2.0046.py
strategy_iter12_sharpe1.7819.py
strategy_iter13_sharpe1.7885.py
strategy_iter15_sharpe1.9058.py
strategy_iter4_sharpe1.6192.py
strategy_iter91_sharpe1.9242.py
